# Regression Analysis of penguin data
## Importing libraries and creating a NaN filtered dataset

In [ ]:

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.axes as ax
import scipy
import statsmodels.formula.api as smf
import statsmodels.api as sm
import pandas as pd

penguins = sns.load_dataset("penguins").dropna() # dopna removes rows with NaN data points

penguins.describe()


## Check models with variables of each non-categorical variable

In [ ]:

def plotPenguins(
    data: pd.Dataframe, fig: plt.Figure, axes: np.ndarray,
    rowIndex: int,  variables: list[str], name: str):

    sns.set()
    for i, var in enumerate(variables):
        model = smf.ols("body_mass_g ~" + var, data=data)
        result = model.fit()
        print(name + " " + var + "'s R²: " + str(result.rsquared))

        if (rowIndex == None):
            sns.scatterplot(
                data=data,
                x=var,
                y="body_mass_g",
                ax=axes[i],
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species"
                )
            print(result.summary())
        else:
            sns.scatterplot(
                data=data,
                x=var,
                y="body_mass_g",
                ax=axes[rowIndex][i],
                hue="sex",
                palette = {
                  "Male": "tab:blue",
                  "Female": "tab:pink",
                },
                style="species"
                )

variables = ["flipper_length_mm", "bill_length_mm", "bill_depth_mm"]
fig, axes = plt.subplots(1,len(variables), figsize=(15,4))
filename = "CDV"
plotPenguins(penguins, fig, axes, None, variables, filename)
fig.savefig(filename + ".jpg")
plt.show()


Flipper length seems to be the best variable for body mass, as the correlation seems highest, and the R² value closest to 1.
It can also be seen that both bill length and depth varies widely based on the species of penguin, which might be the reason for their low R² values. One can also see that the reason flipper length has a comparatively high R² value is that longer flipper length correlates with species and sex with higher body mass, which could be the real variables.
With $\alpha=0.5$, it seems the variables do affect body_mass_g in some way. Notably, the bill depth has an inverse correlation with body mass, maybe because the species that have deeper bills tend to be smaller.
The formulas for flipper_length_mm ~ 

## Filtering by species and sex

In [ ]:

penguinSpecies = ["Adelie", "Chinstrap", "Gentoo"]
sexes = ["Male", "Female"]
fig, axes = plt.subplots(len(sexes)*len(penguinSpecies),len(variables), figsize=(15,30))
for i, species in enumerate(penguinSpecies):
    for j, sex in enumerate(sexes):
        sortedPenguin = penguins[penguins["species"] == species]
        sortedPenguin = sortedPenguin[sortedPenguin["sex"] == sex]
        plotPenguins(sortedPenguin, fig, axes, 2*i+j, variables, species + "-" + sex)
plt.plot()
fig.savefig("CDV-filtered.jpg")


Here we can see that within the categories of sex within species, none of the dependency variables seem useful. This confirms the theory that flipper_length is a useful variable only due to its correlation with sex and species.

## More useful plots
Despite this, we can see that a linear regression model fits the data well, and that we don't need to transform any variables.

In [ ]:

filename = "linearRegression"
fig, axes = plt.subplots(1,1, figsize=(5,5))
sns.regplot(x=penguins["flipper_length_mm"], y=penguins["body_mass_g"], line_kws={"color": "black"}, scatter_kws={"s": 2})
plt.plot()
